In [ ]:
!pip install gradio pypdf python-docx
!pip install gradio pypdf python-docx PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.1 MB/s eta 0:00:00


In [ ]:
import gradio as gr
from PyPDF2 import PdfReader, PdfWriter
from docx import Document
import os

def delete_pdf_pages(file, pages):
    try:
        reader = PdfReader(file.name)
        writer = PdfWriter()

        # Convert user input to list of pages to delete
        delete_pages = set()
        for part in pages.split(","):
            if "-" in part:
                start, end = map(int, part.split("-"))
                delete_pages.update(range(start-1, end))  # Pages are 0-indexed
            else:
                delete_pages.add(int(part)-1)

        for i, page in enumerate(reader.pages):
            if i not in delete_pages:
                writer.add_page(page)

        output_path = "output.pdf"
        with open(output_path, "wb") as output_file:
            writer.write(output_file)

        return output_path
    except Exception as e:
        return str(e)

def delete_docx_pages(file, pages):
    try:
        doc = Document(file.name)
        delete_pages = set()

        for part in pages.split(","):
            if "-" in part:
                start, end = map(int, part.split("-"))
                delete_pages.update(range(start-1, end))  # Pages are 0-indexed
            else:
                delete_pages.add(int(part)-1)

        paragraphs = doc.paragraphs
        new_doc = Document()

        current_page = 0
        for para in paragraphs:
            if current_page not in delete_pages:
                new_doc.add_paragraph(para.text)
            current_page += 1

        output_path = "output.docx"
        new_doc.save(output_path)
        return output_path
    except Exception as e:
        return str(e)

def process_file(file, pages):
    file_extension = file.name.split(".")[-1].lower()
    if file_extension == "pdf":
        return delete_pdf_pages(file, pages)
    elif file_extension in ["doc", "docx"]:
        return delete_docx_pages(file, pages)
    else:
        return "Unsupported file format!"

with gr.Blocks() as demo:
    gr.Markdown("# PDF Page Remover")
    file_input = gr.File(label="Upload your PDF file")
    pages_input = gr.Textbox(label="Enter pages to delete (e.g., 1-5, 7, 10-12)")
    output_file = gr.File(label="Download Processed File")
    submit_button = gr.Button("Process File")
    submit_button.click(process_file, inputs=[file_input, pages_input], outputs=output_file)

demo.launch()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.

To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>